In [ ]:
# Librerías
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.preprocessing import MinMaxScaler
from ucimlrepo import fetch_ucirepo 
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
# ===== CAPA LEGENDRE =====
class PolynomialLegendre(tf.keras.layers.Layer):
    def __init__(self, units, degree=2, use_bias=True, **kwargs):
        super(PolynomialLegendre, self).__init__(**kwargs)
        self.units = units
        self.degree = degree
        self.use_bias = use_bias

    def build(self, input_shape):
        input_dim_leg = input_shape[-1]

        self.kernel_leg = self.add_weight(
            shape=(input_dim_leg * self.degree, self.units),
            initializer=tf.keras.initializers.GlorotUniform(),
            trainable=True,
            name="kernel_leg"
        )

        if self.use_bias:
            self.bias_leg = self.add_weight(
                shape=(self.units,),
                initializer="zeros",
                trainable=True,
                name="bias_leg"
            )

    def call(self, inputs):
        x_leg = tf.cast(inputs, self.compute_dtype)
        
        p_nm2_leg = tf.ones_like(x_leg)
        p_nm1_leg = x_leg
        
        features_leg = [p_nm1_leg]

        for n in range(2, self.degree + 1):
            n_float_leg = tf.cast(n, self.compute_dtype)

            p_n_leg = ((2.0 * n_float_leg - 1.0) * x_leg * p_nm1_leg - (n_float_leg - 1.0) * p_nm2_leg) / n_float_leg
            features_leg.append(p_n_leg)
            
            p_nm2_leg = p_nm1_leg
            p_nm1_leg = p_n_leg

        phi_leg = tf.concat(features_leg, axis=-1)

        output_leg = tf.matmul(phi_leg, self.kernel_leg)

        if self.use_bias:
            output_leg = tf.nn.bias_add(output_leg, self.bias_leg)

        return output_leg


In [ ]:
# ===== PLOT =====
def plot_cv_average_history_leg(histories_leg):
    max_epochs_leg = max([len(h.history['loss']) for h in histories_leg])
    epochs_leg = np.arange(1, max_epochs_leg + 1)

    def get_padded_metrics_leg(metric_name):
        matrix_leg = np.full((len(histories_leg), max_epochs_leg), np.nan)
        for i, h in enumerate(histories_leg):
            data_leg = h.history[metric_name]
            matrix_leg[i, :len(data_leg)] = data_leg
        return np.nanmean(matrix_leg, axis=0)

    avg_loss_leg = get_padded_metrics_leg('loss')
    avg_val_loss_leg = get_padded_metrics_leg('val_loss')
    avg_acc_leg = get_padded_metrics_leg('accuracy')
    avg_val_acc_leg = get_padded_metrics_leg('val_accuracy')

    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_leg, avg_loss_leg)
    plt.plot(epochs_leg, avg_val_loss_leg)
    plt.title('Pérdida Promedio')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_leg, avg_acc_leg)
    plt.plot(epochs_leg, avg_val_acc_leg)
    plt.title('Precisión Promedio')

    plt.tight_layout()
    plt.show()


In [ ]:
# ===== MODELO =====
def PolynomialDenseCreator_leg(degree_leg, input_dim_leg):
    inputPoli_leg = keras.Input(shape=(input_dim_leg,))
    
    x_leg = PolynomialLegendre(32, degree=degree_leg)(inputPoli_leg)
    x_leg = layers.Activation('swish')(x_leg)
    x_leg = layers.Dense(16, activation='swish')(x_leg)
    
    outputPoli_leg = layers.Dense(2, activation='softmax')(x_leg)
    
    model_leg = keras.Model(
        inputs=inputPoli_leg,
        outputs=outputPoli_leg,
        name=f"Polynomial_Model_Degree_{degree_leg}_leg"
    )
    
    model_leg.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model_leg


In [ ]:
# ===== DATOS =====
magic_gamma_telescope_leg = fetch_ucirepo(id=159)

X_leg = magic_gamma_telescope_leg.data.features 
y_leg = magic_gamma_telescope_leg.data.targets 


In [ ]:

# ===== HIPERPARÁMETROS =====
epochs_leg = 10
batch_size_leg = 32
input_dim_leg = X_leg.shape[1]
num_splits_leg = 10
degrees = [3, 5, 7]


In [ ]:
def createEarlyStoppingCallback_leg(patience_leg=15):
    return keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=patience_leg,
        restore_best_weights=True
    )

In [ ]:
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

skf_leg = StratifiedKFold(n_splits=num_splits_leg, shuffle=True, random_state=1)

history_leg = {deg: [] for deg in degrees}
score_leg = {deg: [] for deg in degrees}

X_leg = X_leg.to_numpy()
y_leg = y_leg.to_numpy()


for train_index, test_index in tqdm(skf_leg.split(X_leg, y_leg), total=num_splits_leg):

    X_train_leg, X_test_leg = X_leg[train_index], X_leg[test_index]
    y_train_leg, y_test_leg = y_leg[train_index], y_leg[test_index]

    scaler_leg = MinMaxScaler(feature_range=(-1, 1))
    X_train_scaled_leg = scaler_leg.fit_transform(X_train_leg)
    X_test_scaled_leg = scaler_leg.transform(X_test_leg)

    y_train_leg = (y_train_leg == 'g').astype(int)
    y_test_leg = (y_test_leg == 'g').astype(int)


    for deg in degrees:
        tf.keras.backend.clear_session()

        model = PolynomialDenseCreator_leg(deg, input_dim_leg=X_train_scaled_leg.shape[1])
        TranedModel = model.fit(X_train_scaled_leg, y_train_leg, validation_split=0.2, epochs=epochs_leg, batch_size=32, verbose=0)
        eval_result = model.evaluate(X_test_scaled_leg, y_test_leg, verbose=0)
        
        history_leg[deg].append(TranedModel)
        score_leg[deg].append(eval_result)




In [ ]:

def calculator_leg(scores_leg):
    Totalloss_leg, Totalaccuracy_leg = 0, 0
    for loss, accuracy in scores_leg:
        Totalloss_leg += loss
        Totalaccuracy_leg += accuracy

    return Totalloss_leg/num_splits_leg, Totalaccuracy_leg/num_splits_leg


print("\n" + "="*40)
print("  RESULTADOS FINALES (Promedio CV - Legendre)")
print("="*40)

scoreMean_leg = {}
for deg in degrees:
    scoreMean_leg[deg] = calculator_leg(score_leg[deg])
    print(f"Grado {deg}: Pérdida Promedio = {scoreMean_leg[deg][0]:.4f}, Precisión Promedio = {scoreMean_leg[deg][1]:.4f}")
    plot_cv_average_history_leg(history_leg[deg])


In [10]:
import pandas as pd
import os

def save_results_to_csv_Legendre(scoreMean_dict, degrees_list, filename="temp_res_legendre.csv"):

    # 1. Definir la ruta de la carpeta (un nivel arriba '..', carpeta 'resultados')
    carpeta_destino = os.path.join("..", "resultados")
    
    # 3. Crear la ruta completa del archivo
    ruta_completa = os.path.join(carpeta_destino, filename)
    
    data = []
    
    # 4. Extraer los datos del diccionario scoreMean_leg
    for deg in degrees_list:
        avg_loss = scoreMean_dict[deg][0]
        avg_accuracy = scoreMean_dict[deg][1]
        
        data.append({
            "Polinomio": "Legendre",
            "Grado": deg,
            "Mejor_N": "N/A",  # Legendre no usa el parámetro N de Shmaliy
            "Loss_Promedio": round(avg_loss, 6),
            "Accuracy_Promedio": round(avg_accuracy, 6)
        })
        
    # 5. Crear DataFrame y guardar a CSV
    df_resultados = pd.DataFrame(data)
    df_resultados.to_csv(ruta_completa, index=False, sep=';')
    
    print(f"\n[INFO] Resultados de Legendre guardados en:\n ---> '{os.path.abspath(ruta_completa)}'")

# ===== LLAMADA A LA FUNCIÓN =====
# Ejecuta esto al final de tu notebook, después de que se calculen los promedios
save_results_to_csv_Legendre(scoreMean_leg, degrees, filename="temp_res_legendre.csv")


[INFO] Resultados de Legendre guardados en:
 ---> 'c:\Users\nicol\Desktop\Universidad\TFG\resultados\temp_res_legendre.csv'
